In [1]:
!pip install transformers torch datasets -q



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\rautr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load cleaned dataset
df = pd.read_csv("../data/processed/cleaned_complaints.csv")

# Keep only required columns
df = df[["clean_text", "category"]]

# Remove missing values (safety)
df = df.dropna()

# Encode labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["category"])

# Check data
print(df.head())
print("Number of classes:", len(label_encoder.classes_))


                                          clean_text  \
0  summer xx xx denied mortgage loan due charge x...   
1  many mistakes appear report without understanding   
2  many mistakes appear report without understanding   
3  many mistakes appear report without understanding   
4  many mistakes appear report without understanding   

                                            category  label  
0  Credit reporting, credit repair services, or o...      2  
1  Credit reporting, credit repair services, or o...      2  
2  Credit reporting, credit repair services, or o...      2  
3  Credit reporting, credit repair services, or o...      2  
4  Credit reporting, credit repair services, or o...      2  
Number of classes: 9


In [2]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


Training samples: 3488
Validation samples: 872


In [3]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization completed")


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tokenization completed


In [4]:
import torch
from torch.utils.data import Dataset

class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

print("Datasets ready for DistilBERT training")


Datasets ready for DistilBERT training


In [5]:
from transformers import DistilBertForSequenceClassification

num_labels = len(set(train_labels))

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("DistilBERT model loaded with", num_labels, "labels")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model loaded with 9 labels


In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Trainer setup completed")


Trainer setup completed


In [7]:
trainer.train()


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.747600
100,1.587800
150,1.385200
200,1.327200
250,1.080300
300,1.004800
350,0.912200
400,0.796300
450,0.840500
500,0.791300


TrainOutput(global_step=872, training_loss=0.9527433673176197, metrics={'train_runtime': 1822.227, 'train_samples_per_second': 3.828, 'train_steps_per_second': 0.479, 'total_flos': 231051983044608.0, 'train_loss': 0.9527433673176197, 'epoch': 2.0})

In [8]:
metrics = trainer.evaluate()
print(metrics)


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7073612213134766, 'eval_runtime': 58.5021, 'eval_samples_per_second': 14.905, 'eval_steps_per_second': 1.863, 'epoch': 2.0}


In [4]:
from transformers import DistilBertTokenizerFast
import torch

# Load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

# Tokenize text
encodings = tokenizer(
    df["clean_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

# Convert labels to tensor
labels = torch.tensor(df["label"].values)

print("Tokenization completed")
print("Sample input_ids length:", len(encodings["input_ids"][0]))


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rautr\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingfa

Tokenization completed
Sample input_ids length: 128


In [5]:
from sklearn.model_selection import train_test_split

# Train-test split (BERT style)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["clean_text"].tolist(),
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))


Training samples: 3488
Validation samples: 872


In [6]:
# Tokenize train and validation text
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    val_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Train & Validation tokenization done")


Train & Validation tokenization done


In [7]:
from torch.utils.data import Dataset

class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)

print("Dataset ready for DistilBERT training")


Dataset ready for DistilBERT training


In [8]:
from transformers import DistilBertForSequenceClassification

num_labels = len(set(train_labels))

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

print("DistilBERT model loaded with", num_labels, "labels")


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model loaded with 3488 labels


In [13]:
!pip install -U accelerate transformers torch



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: C:\Users\rautr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import torch
import transformers
import accelerate

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)


C:\Users\rautr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Torch: 2.9.1+cpu
Transformers: 4.57.3
Accelerate: 1.12.0
